# ChuckleNet WavLM + Prosody Extraction v10 (FIXED) (FINAL)

**Auto-fetches utterances from Drive, uses existing MP3/WAV audio**

- ✅ Auto-loads `utterances_clean.jsonl` from Drive (no upload needed)
- ✅ Searches ALL audio subfolders (MP3 + WAV)
- ✅ Skips .part (incomplete) files
- ✅ 768-dim WavLM (attention pooling)
- ✅ 21-dim prosody
- ✅ Train/Val/Test by comedian
- ✅ Progress + checkpoint every 1000

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

## Step 2: Install Packages

In [ ]:
!pip install -q transformers librosa scikit-learn
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Packages installed!')

## Step 3: Load Utterances (AUTO from Drive)

In [ ]:
# Step 3: Download utterances_clean.jsonl from Google Drive
import os
import subprocess

UTTERANCES_PATH = '/content/gdrive/MyDrive/utterances_clean.jsonl'

if not os.path.exists(UTTERANCES_PATH):
    print('📥 Downloading utterances_clean.jsonl...')
    FILE_ID = '1cuhs6mh-r9Spzq9cTDG8AidT53DLsALn'
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True, capture_output=True)
    subprocess.run([
        'gdown', '--fuzzy', '-O', UTTERANCES_PATH,
        f'https://drive.google.com/file/d/{FILE_ID}/view'
    ], check=True, capture_output=True)
    print('✅ Downloaded utterances_clean.jsonl')
else:
    print('✅ Found utterances_clean.jsonl')

# Load utterances
import json
print(f'📋 Loading utterances...')
utterances = []
with open(UTTERANCES_PATH, 'r') as f:
    for line in f:
        utterances.append(json.loads(line.strip()))
print(f'✅ Loaded {len(utterances)} utterances')

## Step 4: Build Audio Map (ALL subfolders, MP3 + WAV)

In [ ]:
# Build audio file map from CORRECT folders only
import os
from pathlib import Path

BASE_DIR = Path('/content/gdrive/MyDrive')

# ONLY search these specific folders - DON'T use broad recursive search
AUDIO_FOLDERS = [
    'chuckle_audio',
    'chuckle_audio_all/audio',
    'chuckle_audio_all/audio_final',
    'chuckle_audio_all/audio_new',
]

print(f"🔍 Searching ONLY correct audio folders...")

audio_files = {}
skipped_incomplete = 0

for folder in AUDIO_FOLDERS:
    audio_dir = BASE_DIR / folder
    if audio_dir.exists():
        files = list(audio_dir.iterdir())
        mp3_files = [p for p in files if p.suffix == '.mp3' and not p.name.endswith('.part')]
        wav_files = [p for p in files if p.suffix == '.wav' and not p.name.endswith('.part')]
        part_files = [p for p in files if p.name.endswith('.part')]
        
        print(f"  📁 {folder}: MP3={len(mp3_files)}, WAV={len(wav_files)}, .part={len(part_files)}")
        
        for p in mp3_files + wav_files:
            audio_files[p.stem] = str(p)
        skipped_incomplete += len(part_files)
    else:
        print(f"  ❌ {folder}: NOT FOUND")

print(f'\n🎵 Total audio files mapped: {len(audio_files)}')
if skipped_incomplete > 0:
    print(f'⚠️ Skipped .part files: {skipped_incomplete}')

# Load utterances (already done in Step 3)
print(f'📋 Loaded {len(utterances)} utterances (from Step 3)')

# Filter to valid utterances  
valid_utterances = [u for u in utterances if u['video_id'] in audio_files]
print(f'✅ Valid utterances with audio: {len(valid_utterances)} / {len(utterances)}')

if len(valid_utterances) == 0:
    print(f'\n❌ ERROR: No valid utterances!')
    print(f'\n🔍 Debug - checking some samples:')
    sample_vid = utterances[0]['video_id'] if utterances else 'N/A'
    print(f'   Sample video_id: {sample_vid}')
    print(f'   Sample audio stems: {list(audio_files.keys())[:5]}')
    if sample_vid in audio_files:
        print(f'   ✅ "{sample_vid}" IS in audio_files!')
    else:
        print(f'   ❌ "{sample_vid}" NOT in audio_files')
        # Try to find similar
        for stem in list(audio_files.keys())[:5]:
            print(f'   Available stem: {stem}')
else:
    # Show split info
    train_count = sum(1 for u in valid_utterances if u.get('split', 'train') == 'train')
    val_count = sum(1 for u in valid_utterances if u.get('split', 'train') == 'val')
    test_count = sum(1 for u in valid_utterances if u.get('split', 'train') == 'test')
    print(f'\n📊 Current split: Train={train_count}, Val={val_count}, Test={test_count}')

## Step 5: 21-dim Prosody Extractor

In [ ]:
import numpy as np
import librosa

def extract_prosody_21dim(y, sr):
    """Extract 21 prosody features."""
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0, 0, 0, 0, 0])
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([
        np.mean(rms), np.std(rms), np.max(rms), np.min(rms),
        np.max(rms) - np.min(rms)
    ])
    
    # Duration - 2 dims
    duration = len(y) / sr
    speech_rate = np.sum(voiced_flag) / duration if duration > 0 else 0
    features.extend([duration, speech_rate])
    
    # Spectral - 5 dims
    try:
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=3)
        mfcc_delta = librosa.feature.delta(mfccs)
        spectral_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        features.extend([
            np.mean(mfccs[0]), np.mean(mfccs[1]), np.mean(mfccs[2]),
            np.mean(mfcc_delta),
            np.mean(spectral_cent) / sr * 1000
        ])
    except:
        features.extend([0, 0, 0, 0, 0])
    
    # Voice quality - 4 dims
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([
        np.mean(zcr),
        np.std(f0_clean) / np.mean(f0_clean) if len(f0_clean) > 1 and np.mean(f0_clean) > 0 else 0,
        np.std(rms) / (np.mean(rms) + 1e-8),
        np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
    ])
    
    return np.array(features, dtype=np.float32)  # 21 dims

# Test
test_pros = extract_prosody_21dim(np.random.randn(16000), 16000)
print(f'✅ Prosody: {test_pros.shape[0]} dims')

## Step 6: Load Wav2Vec2

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torch

# Auto-detect device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Using device: {DEVICE}')
if DEVICE == 'cpu':
    print('⚠️ WARNING: CPU is VERY slow for Wav2Vec2. Use T4 GPU runtime!')

MODEL_NAME = 'facebook/wav2vec2-large-xlsr-53'

print('📥 Loading Wav2Vec2 (1.27GB)...')
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
wav2vec = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print(f'✅ Model loaded on {DEVICE}!')

## Step 7: Split by Comedian

In [ ]:
import random
import numpy as np

# Random 80/10/10 split (no comedian info available)
random.seed(42)
np.random.seed(42)

video_ids = list({u['video_id'] for u in valid_utterances})
random.shuffle(video_ids)

n = len(video_ids)
val_size = int(n * 0.1)
test_size = int(n * 0.1)

val_vids = set(video_ids[:val_size])
test_vids = set(video_ids[val_size:val_size + test_size])

def get_split(video_id):
    if video_id in val_vids:
        return 'val'
    elif video_id in test_vids:
        return 'test'
    return 'train'

for u in valid_utterances:
    u['split'] = get_split(u['video_id'])

for s, c in [('train', sum(1 for u in valid_utterances if u['split']=='train')),
             ('val', sum(1 for u in valid_utterances if u['split']=='val')),
             ('test', sum(1 for u in valid_utterances if u['split']=='test'))]:
    print(f'   {s}: {c} ({c/len(valid_utterances)*100:.1f}%)')


## Step 8: Extract Features (Progress + Checkpoints)

In [ ]:
import numpy as np
import librosa
import time, os, json

SR = 16000
BATCH_SIZE = 32
CHECKPOINT_DIR = '/content/gdrive/MyDrive/wavlm_extraction_v6_checkpoints'
CHECKPOINT_INTERVAL = 1000
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load checkpoint if exists
ckpt_file = f'{CHECKPOINT_DIR}/checkpoint.json'
if os.path.exists(ckpt_file):
    with open(ckpt_file) as f:
        ckpt = json.load(f)
    start_idx = ckpt['last_idx'] + 1
    train_emb = ckpt.get('train_emb', [])[-10000:]
    train_pros = ckpt.get('train_pros', [])[-10000:]
    train_lbl = ckpt.get('train_lbl', [])[-10000:]
    val_emb = ckpt.get('val_emb', [])
    val_pros = ckpt.get('val_pros', [])
    val_lbl = ckpt.get('val_lbl', [])
    test_emb = ckpt.get('test_emb', [])
    test_pros = ckpt.get('test_pros', [])
    test_lbl = ckpt.get('test_lbl', [])
    print(f'📂 Resume from idx {start_idx} | T:{len(train_emb)} V:{len(val_emb)} Te:{len(test_emb)}')
else:
    start_idx = 0
    train_emb, train_pros, train_lbl = [], [], []
    val_emb, val_pros, val_lbl = [], [], []
    test_emb, test_pros, test_lbl = [], [], []
    print('🆕 Fresh extraction')

total = len(valid_utterances)
t0 = time.time()
skipped = 0

for batch_start in range(start_idx, total, BATCH_SIZE):
    batch = valid_utterances[batch_start:batch_start + BATCH_SIZE]
    
    for u in batch:
        audio_path = audio_files.get(u['video_id'])
        if not audio_path:
            skipped += 1
            continue
        
        try:
            # librosa handles both MP3 and WAV
            y, sr = librosa.load(audio_path, sr=SR, mono=True,
                               offset=u['start'],
                               duration=u['end']-u['start'])
            if len(y) < SR * 0.1:
                skipped += 1
                continue
            
            # Wav2Vec2
            inputs = feature_extractor(y, sampling_rate=SR, return_tensors='pt', padding=True)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = wav2vec(**inputs)
            
            # Attention pooling
            hidden = outputs.last_hidden_state
            attn = torch.softmax(torch.nn.Linear(768, 1)(hidden).squeeze(-1), dim=-1)
            emb = torch.bmm(attn.unsqueeze(1), hidden).squeeze(1).squeeze(0).cpu().numpy()
            
            # 21-dim prosody
            prosody = extract_prosody_21dim(y, sr)
            
            # Store by split
            split = u['split']
            if split == 'train':
                train_emb.append(emb); train_pros.append(prosody); train_lbl.append(u['label'])
            elif split == 'val':
                val_emb.append(emb); val_pros.append(prosody); val_lbl.append(u['label'])
            else:
                test_emb.append(emb); test_pros.append(prosody); test_lbl.append(u['label'])
                
        except Exception as e:
            skipped += 1
            continue
    
    # Progress
    idx = batch_start + BATCH_SIZE
    elapsed = time.time() - t0
    rate = idx / elapsed if elapsed > 0 else 0
    eta = (total - idx) / rate / 60 if rate > 0 else 0
    pos = sum(train_lbl) + sum(val_lbl) + sum(test_lbl)
    n = len(train_lbl) + len(val_lbl) + len(test_lbl)
    
    print(f'📊 {idx}/{total} ({(idx/total*100):.1f}%) | {rate:.1f}/s | ETA:{eta:.0f}min | '
          f'T:{len(train_emb)} V:{len(val_emb)} Te:{len(test_emb)} | '
          f'Pos:{pos/max(n,1)*100:.1f}% | Skip:{skipped}')
    
    # Checkpoint
    if idx % CHECKPOINT_INTERVAL == 0 or idx >= total:
        with open(ckpt_file, 'w') as f:
            json.dump({
                'last_idx': idx - 1,
                'train_emb': train_emb[-10000:],
                'train_pros': train_pros[-10000:],
                'train_lbl': train_lbl[-10000:],
                'val_emb': val_emb, 'val_pros': val_pros, 'val_lbl': val_lbl,
                'test_emb': test_emb, 'test_pros': test_pros, 'test_lbl': test_lbl,
                'timestamp': time.time()
            }, f)
        print(f'💾 Checkpoint saved')

print(f'\n✅ Done! T:{len(train_emb)} V:{len(val_emb)} Te:{len(test_emb)} | Skipped:{skipped}')

## Step 9: Save Results

In [ ]:
OUT = '/content/gdrive/MyDrive/wavlm_prosody_v6'
os.makedirs(OUT, exist_ok=True)

np.savez(f'{OUT}/train.npz', 
          wavlm=np.array(train_emb, dtype=np.float32),
          prosody=np.array(train_pros, dtype=np.float32),
          labels=np.array(train_lbl, dtype=np.int32))
np.savez(f'{OUT}/val.npz',
         wavlm=np.array(val_emb, dtype=np.float32),
         prosody=np.array(val_pros, dtype=np.float32),
         labels=np.array(val_lbl, dtype=np.int32))
np.savez(f'{OUT}/test.npz',
         wavlm=np.array(test_emb, dtype=np.float32),
         prosody=np.array(test_pros, dtype=np.float32),
         labels=np.array(test_lbl, dtype=np.int32))

print(f'✅ Saved to {OUT}')
print(f'   Train: {len(train_emb)} x WavLM{list(np.array(train_emb[0]).shape) if train_emb else "?"} + Prosody21')
print(f'   Val:   {len(val_emb)}')
print(f'   Test:  {len(test_emb)}')